In [0]:
from pyspark.sql import functions as F

members_path = "/Volumes/workspace/kkbox/raw_data/members_v3.csv"

members_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(members_path)
)

display(
    members_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("msno").alias("unique_customers"),
        F.sum(F.when(F.col("msno").isNull(), 1).otherwise(0)).alias("null_customer_ids"),
        F.sum(F.when(F.col("city").isNull(), 1).otherwise(0)).alias("null_city"),
        F.sum(F.when(F.col("bd").isNull(), 1).otherwise(0)).alias("null_age"),
        F.sum(F.when(F.col("gender").isNull(), 1).otherwise(0)).alias("null_gender"),
        F.sum(F.when(F.col("registered_via").isNull(), 1).otherwise(0)).alias("null_registered_via"),
        F.sum(F.when(F.col("registration_init_time").isNull(), 1).otherwise(0)).alias("null_registration_date")
    )
)

In [0]:
display(
    members_df
    .groupBy("gender")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    members_df
    .select(
        F.min("bd").alias("min_age"),
        F.max("bd").alias("max_age"),
        F.avg("bd").alias("average_age"),
        F.expr("percentile_approx(bd, 0.5)").alias("median_age")
    )
)

In [0]:
from pyspark.sql import functions as F

age_quality = (
    members_df
    .agg(
        F.sum(
            F.when((F.col("bd") < 10) | (F.col("bd") > 100), 1)
            .otherwise(0)
        ).alias("invalid_age_count"),

        F.sum(
            F.when((F.col("bd") >= 10) & (F.col("bd") <= 100), 1)
            .otherwise(0)
        ).alias("valid_age_count"),

        F.sum(
            F.when(F.col("bd") == 0, 1)
            .otherwise(0)
        ).alias("zero_age_count")
    )
)

display(age_quality)

In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table("workspace.kkbox.customer_scope")

members_clean = (
    customer_scope
    .join(
        members_df,
        on="msno",
        how="left"
    )
    .select(
        "msno",
        
        F.col("city").alias("city"),
        
        F.when(
            (F.col("bd") >= 10) & (F.col("bd") <= 100),
            F.col("bd")
        ).otherwise(None).alias("age"),
        
        F.when(
            F.col("gender").isin("male", "female"),
            F.col("gender")
        ).otherwise("unknown").alias("gender"),
        
        F.col("registered_via").alias("registered_via"),
        
        F.to_date(
            F.col("registration_init_time").cast("string"),
            "yyyyMMdd"
        ).alias("registration_date"),
        
        F.when(
            F.col("city").isNotNull(),
            1
        ).otherwise(0).alias("has_member_data")
    )
)

display(members_clean.limit(10))

In [0]:
display(
    members_clean.agg(
        F.count("*").alias("total_customers"),
        F.countDistinct("msno").alias("unique_customers"),
        F.sum(
            F.when(F.col("age").isNull(), 1).otherwise(0)
        ).alias("unknown_age"),
        F.sum(
            F.when(F.col("gender") == "unknown", 1).otherwise(0)
        ).alias("unknown_gender"),
        F.sum("has_member_data").alias("customers_with_member_data"),
        F.sum(
            F.when(F.col("has_member_data") == 0, 1).otherwise(0)
        ).alias("customers_missing_member_data")
    )
)

In [0]:
members_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.members_clean")

In [0]:
display(
    spark.table("workspace.kkbox.members_clean")
    .agg(
        F.count("*").alias("total_customers"),
        F.countDistinct("msno").alias("unique_customers")
    )
)